## The purpose of this file
1)  merge the two voter registration files into one file
2)  merge that file into the election votig 

**COMBINE REGISTRATION FILES**

In [8]:
import pandas as pd

reg20_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/processed/VoterReg2020_cleaned_v2.csv"
reg24_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/processed/VoterReg2024_cleaned.csv"
reg_out_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/processed/voter_registration_combined.csv"

reg20 = pd.read_csv(reg20_path)
reg24 = pd.read_csv(reg24_path)

reg20["county"] = reg20["county"].astype(str).str.strip()
reg24["county"] = reg24["county"].astype(str).str.strip()

reg20["year"] = 2020
reg24["year"] = 2024

reg24["other_parties"] = pd.to_numeric(reg24["other_parties"], errors="coerce").fillna(0) + pd.to_numeric(reg24["libertarian"], errors="coerce").fillna(0)
reg24 = reg24.drop(columns=["libertarian"])

**RENAME THE REGISTRATION COLUMNS**

In [9]:
reg20 = reg20.rename(columns={
    reg20.columns[2]: "democrat_reg",
    reg20.columns[3]: "republican_reg",
    reg20.columns[4]: "other_parties_reg",
    reg20.columns[5]: "all_parties_reg"
})

reg24 = reg24.rename(columns={
    reg24.columns[2]: "democrat_reg",
    reg24.columns[3]: "republican_reg",
    reg24.columns[4]: "other_parties_reg",
    reg24.columns[5]: "all_parties_reg"
})

**COMBINE AND SORT THE REGISTRATION DATA**

In [10]:
registration = pd.concat([reg20, reg24], ignore_index=True)
registration = registration.sort_values(["county", "year"]).reset_index(drop=True)

registration.to_csv(reg_out_path, index=False)

registration.head()

,county,year,democrat_reg,republican_reg,other_parties_reg,all_parties_reg
0,Adams,2020,20063,40781,11544.0,72388
1,Adams,2024,19474,43914,12620.0,76008
2,Allegheny,2020,539595,267832,134829.0,942256
3,Allegheny,2024,526268,274157,152118.0,952543
4,Armstrong,2020,13659,25979,5188.0,44826


**MERGE REGISTRATION TO ELECTION DATA**

In [15]:
election_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/processed/a_countypres_2020-2024_partytotals.csv"
final_out_path = r"C:/Users/mutch/Democratic_Voter_Shift/data/processed/election_with_registration.csv"

election = pd.read_csv(election_path)

if "county_name" in election.columns:
    election = election.rename(columns={"county_name": "county"})
else:
    raise KeyError("Expected column 'county_name' was not found in election file.")

election["county"] = election["county"].astype(str).str.strip().str.title()
election["year"] = election["year"].astype(int)

registration["county"] = registration["county"].astype(str).str.strip().str.title()
registration["year"] = registration["year"].astype(int)

final_df = election.merge(registration, on=["county", "year"], how="left", validate="1:1")
final_df = final_df.sort_values(["county", "year"]).reset_index(drop=True)

final_df.columns = final_df.columns.str.lower()

if "libertarian" in final_df.columns and "other" in final_df.columns:
    final_df["other"] = pd.to_numeric(final_df["other"], errors="coerce").fillna(0) + pd.to_numeric(final_df["libertarian"], errors="coerce").fillna(0)
    final_df = final_df.drop(columns=["libertarian"])
else:
    raise KeyError("Expected columns 'libertarian' and 'other' were not found.")

final_df.to_csv(final_out_path, index=False)

final_df.head()



,county,year,state_po,county_fips,totalvotes,democrat,other,republican,democrat_reg,republican_reg,other_parties_reg,all_parties_reg
0,Adams,2020,PA,42001.0,56540,18207.0,810.0,37523.0,20063.0,40781,11544.0,72388
1,Adams,2024,PA,42001.0,60650,19842.0,560.0,40248.0,19474.0,43914,12620.0,76008
2,Allegheny,2020,PA,42003.0,719733,429065.0,8344.0,282324.0,539595.0,267832,134829.0,942256
3,Allegheny,2024,PA,42003.0,720504,429916.0,6993.0,283595.0,526268.0,274157,152118.0,952543
4,Armstrong,2020,PA,42005.0,36370,8457.0,424.0,27489.0,13659.0,25979,5188.0,44826
